In [0]:
%sql
CREATE CATALOG IF NOT EXISTS global_weather_catalog;

In [0]:
%sql
USE CATALOG global_weather_catalog;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
USE CATALOG global_weather_catalog;
USE SCHEMA gold;

In [0]:
%sql
SELECT COUNT(*) AS total_rows FROM global_weather_catalog.silver.weather_silver;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import datetime
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("gold_weather")

BUCKET    = "s3://global-weather-pipeline"
CATALOG   = "global_weather_catalog"
GOLD_PATH = f"{BUCKET}/gold/"
batch_date = datetime.date.today().strftime("%Y-%m-%d")

# ── Read Silver table ──────────────────────────────────────
df_silver = spark.table(f"{CATALOG}.silver.weather_silver")
count = df_silver.count()

print(f"✅ Silver data loaded!")
print(f"   Total rows    : {count}")
print(f"   Total columns : {len(df_silver.columns)}")
print(f"\n📋 Columns available:")
for c in df_silver.columns:
    print(f"   {c}")

In [0]:
# ── dim_location ───────────────────────────────────────────
# Contains WHERE each weather reading came from.
# Location data is static — same city always has same
# lat/long/timezone so no need to repeat in fact table.

dim_location = df_silver.select(
    "weather_record_id",
    "country",
    "location_name",
    "latitude",
    "longitude",
    "timezone",
    "region_category",
    "weather_station_name",
    "sensor_type"
).dropDuplicates(["weather_record_id"])

dim_location.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}dim_location/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.dim_location
    USING DELTA
    LOCATION '{GOLD_PATH}dim_location/'
""")

print(f"✅ dim_location created!")
print(f"   Rows    : {dim_location.count()}")
print(f"   Columns : {len(dim_location.columns)}")
print(f"   Columns : {dim_location.columns}")
display(dim_location.limit(5))

In [0]:
# ── dim_condition ──────────────────────────────────────────
# Contains descriptive/qualitative weather labels.
# These are text categories that describe the weather —
# kept separate from numeric measurements in fact table.

dim_condition = df_silver.select(
    "weather_record_id",
    "condition_text",
    "moon_phase",
    "temperature_category",
    "humidity_level",
    "wind_intensity_level",
    "precipitation_level",
    "visibility_category",
    "uv_risk_level",
    "air_quality_category",
    "cloud_cover_level"
).dropDuplicates(["weather_record_id"])

dim_condition.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}dim_condition/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.dim_condition
    USING DELTA
    LOCATION '{GOLD_PATH}dim_condition/'
""")

print(f"✅ dim_condition created!")
print(f"   Rows    : {dim_condition.count()}")
print(f"   Columns : {len(dim_condition.columns)}")
print(f"   Columns : {dim_condition.columns}")
display(dim_condition.limit(5))

In [0]:
# ── dim_astronomy ──────────────────────────────────────────
# Contains sky/astronomy data — sunrise, sunset, moon info.
# Independent of weather sensor readings so kept separate.

dim_astronomy = df_silver.select(
    "weather_record_id",
    "sunrise",
    "sunset",
    "moonrise",
    "moonset",
    "moon_phase",
    "moon_illumination"
).dropDuplicates(["weather_record_id"])

dim_astronomy.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}dim_astronomy/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.dim_astronomy
    USING DELTA
    LOCATION '{GOLD_PATH}dim_astronomy/'
""")

print(f"✅ dim_astronomy created!")
print(f"   Rows    : {dim_astronomy.count()}")
print(f"   Columns : {len(dim_astronomy.columns)}")
print(f"   Columns : {dim_astronomy.columns}")
display(dim_astronomy.limit(5))

In [0]:
# ── dim_date ───────────────────────────────────────────────
# Contains all time-related columns derived from last_updated.
# Standard in every data warehouse — lets analysts slice
# data by hour, day, month, year easily.

dim_date = df_silver.select(
    "weather_record_id",
    "last_updated",
    "date",
    "hour",
    "month",
    "year",
    "day_of_week",
    "is_day"
).dropDuplicates(["weather_record_id"])

dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}dim_date/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.dim_date
    USING DELTA
    LOCATION '{GOLD_PATH}dim_date/'
""")

print(f"✅ dim_date created!")
print(f"   Rows    : {dim_date.count()}")
print(f"   Columns : {len(dim_date.columns)}")
print(f"   Columns : {dim_date.columns}")
display(dim_date.limit(5))

In [0]:
# ── fact_weather ───────────────────────────────────────────
# The central fact table — contains ALL numeric measurements.
# This is what analysts query and aggregate.
# All dims link back here via weather_record_id.

fact_weather = df_silver.select(
    "weather_record_id",
    "last_updated_epoch",
    "temperature_celsius",
    "feels_like_celsius",
    "humidity",
    "wind_kph",
    "wind_degree",
    "wind_direction",
    "pressure_mb",
    "precip_mm",
    "cloud",
    "visibility_km",
    "uv_index",
    "gust_kph",
    "air_quality_pm2_5",
    "air_quality_pm10",
    "air_quality_carbon_monoxide",
    "air_quality_ozone",
    "air_quality_nitrogen_dioxide",
    "air_quality_sulphur_dioxide",
    "air_quality_us_epa_index",
    "air_quality_gb_defra_index",
    "is_outlier",
    "_batch_date",
    "_ingestion_ts",
    "_source_path",
    "_layer"
).dropDuplicates(["weather_record_id"])

fact_weather.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("_batch_date") \
    .save(f"{GOLD_PATH}fact_weather/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.fact_weather
    USING DELTA
    LOCATION '{GOLD_PATH}fact_weather/'
""")

spark.sql(f"OPTIMIZE {CATALOG}.gold.fact_weather ZORDER BY (weather_record_id)")

print(f"✅ fact_weather created!")
print(f"   Rows    : {fact_weather.count()}")
print(f"   Columns : {len(fact_weather.columns)}")
display(fact_weather.limit(55))

In [0]:
# ── Verify all Gold tables created ────────────────────────
tables = ["dim_location", "dim_condition", "dim_astronomy", "dim_date", "fact_weather"]

print(f"📊 Gold Layer — Table Verification")
print(f"{'='*50}")
total_rows = 0
for table in tables:
    count = spark.sql(
        f"SELECT COUNT(*) AS cnt FROM {CATALOG}.gold.{table}"
    ).collect()[0]["cnt"]
    cols = len(spark.table(f"{CATALOG}.gold.{table}").columns)
    print(f"   ✅ {table:<20} → {count:>7} rows, {cols} columns")
    total_rows += count

print(f"\n   Total rows across all Gold tables : {total_rows}")
print(f"   All 5 Gold tables created successfully ✓")

In [0]:
# ── Insight 1: Top 10 Hottest & Coldest Countries ─────────
# Joins fact_weather with dim_location to get country names
# then aggregates average temperature per country.

insight_1 = spark.sql(f"""
    SELECT
        l.country,
        ROUND(AVG(f.temperature_celsius), 2)  AS avg_temp_celsius,
        ROUND(MAX(f.temperature_celsius), 2)  AS max_temp_celsius,
        ROUND(MIN(f.temperature_celsius), 2)  AS min_temp_celsius,
        COUNT(*)                              AS reading_count
    FROM {CATALOG}.gold.fact_weather f
    JOIN {CATALOG}.gold.dim_location l
        ON f.weather_record_id = l.weather_record_id
    WHERE f.is_outlier = false
    GROUP BY l.country
    ORDER BY avg_temp_celsius DESC
""")

# Save as Gold table
insight_1.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}insight_top_temperatures/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.insight_top_temperatures
    USING DELTA
    LOCATION '{GOLD_PATH}insight_top_temperatures/'
""")

print("🌡️ Insight 1: Top 10 HOTTEST Countries:")
display(insight_1.limit(10))

print("🌡️ Insight 1: Top 10 COLDEST Countries:")
display(insight_1.orderBy("avg_temp_celsius").limit(10))

In [0]:
# ── Insight 2: Worst Air Quality Countries (PM2.5) ─────────
insight_2 = spark.sql(f"""
    SELECT
        l.country,
        ROUND(AVG(f.air_quality_pm2_5), 2)          AS avg_pm2_5,
        ROUND(AVG(f.air_quality_pm10), 2)            AS avg_pm10,
        ROUND(AVG(f.air_quality_carbon_monoxide), 2) AS avg_co,
        ROUND(AVG(f.air_quality_ozone), 2)           AS avg_ozone,
        ROUND(AVG(f.air_quality_us_epa_index), 2)    AS avg_epa_index,
        COUNT(*)                                     AS reading_count
    FROM {CATALOG}.gold.fact_weather f
    JOIN {CATALOG}.gold.dim_location l
        ON f.weather_record_id = l.weather_record_id
    WHERE f.is_outlier = false
    GROUP BY l.country
    ORDER BY avg_pm2_5 DESC
""")

insight_2.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}insight_air_quality/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.insight_air_quality
    USING DELTA
    LOCATION '{GOLD_PATH}insight_air_quality/'
""")

print("💨 Insight 2: Top 10 WORST Air Quality Countries (by PM2.5):")
display(insight_2.limit(10))

In [0]:
# ── Insight 3: Highest Rainfall Regions ───────────────────
insight_3 = spark.sql(f"""
    SELECT
        l.country,
        l.location_name,
        ROUND(SUM(f.precip_mm), 2)  AS total_rainfall_mm,
        ROUND(AVG(f.precip_mm), 2)  AS avg_rainfall_mm,
        ROUND(MAX(f.precip_mm), 2)  AS max_rainfall_mm,
        COUNT(*)                    AS reading_count
    FROM {CATALOG}.gold.fact_weather f
    JOIN {CATALOG}.gold.dim_location l
        ON f.weather_record_id = l.weather_record_id
    WHERE f.is_outlier = false
    GROUP BY l.country, l.location_name
    ORDER BY total_rainfall_mm DESC
""")

insight_3.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}insight_rainfall/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.insight_rainfall
    USING DELTA
    LOCATION '{GOLD_PATH}insight_rainfall/'
""")

print("🌧️ Insight 3: Top 10 Highest Rainfall Regions:")
display(insight_3.limit(10))

In [0]:
# ── Insight 4: Windiest Countries ─────────────────────────
insight_4 = spark.sql(f"""
    SELECT
        l.country,
        ROUND(AVG(f.wind_kph), 2)  AS avg_wind_kph,
        ROUND(MAX(f.wind_kph), 2)  AS max_wind_kph,
        ROUND(AVG(f.gust_kph), 2)  AS avg_gust_kph,
        ROUND(MAX(f.gust_kph), 2)  AS max_gust_kph,
        COUNT(*)                   AS reading_count
    FROM {CATALOG}.gold.fact_weather f
    JOIN {CATALOG}.gold.dim_location l
        ON f.weather_record_id = l.weather_record_id
    WHERE f.is_outlier = false
    GROUP BY l.country
    ORDER BY avg_wind_kph DESC
""")

insight_4.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}insight_wind/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.insight_wind
    USING DELTA
    LOCATION '{GOLD_PATH}insight_wind/'
""")

print("💨 Insight 4: Top 10 Windiest Countries:")
display(insight_4.limit(10))

In [0]:
# ── Insight 5: UV Risk Analysis by Country ────────────────
insight_5 = spark.sql(f"""
    SELECT
        l.country,
        ROUND(AVG(f.uv_index), 2)   AS avg_uv_index,
        ROUND(MAX(f.uv_index), 2)   AS max_uv_index,
        c.uv_risk_level,
        COUNT(*)                    AS reading_count
    FROM {CATALOG}.gold.fact_weather f
    JOIN {CATALOG}.gold.dim_location l
        ON f.weather_record_id = l.weather_record_id
    JOIN {CATALOG}.gold.dim_condition c
        ON f.weather_record_id = c.weather_record_id
    WHERE f.is_outlier = false
    GROUP BY l.country, c.uv_risk_level
    ORDER BY avg_uv_index DESC
""")

insight_5.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}insight_uv_risk/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.insight_uv_risk
    USING DELTA
    LOCATION '{GOLD_PATH}insight_uv_risk/'
""")

print("☀️ Insight 5: Top 10 Highest UV Risk Countries:")
display(insight_5.limit(10))

In [0]:
# ── Insight 6: Weather Condition Distribution ──────────────
insight_6 = spark.sql(f"""
    SELECT
        c.condition_text,
        COUNT(*)                                AS total_readings,
        ROUND(COUNT(*) * 100.0 / 
            SUM(COUNT(*)) OVER(), 2)            AS percentage,
        ROUND(AVG(f.temperature_celsius), 2)    AS avg_temp,
        ROUND(AVG(f.humidity), 2)               AS avg_humidity
    FROM {CATALOG}.gold.fact_weather f
    JOIN {CATALOG}.gold.dim_condition c
        ON f.weather_record_id = c.weather_record_id
    WHERE f.is_outlier = false
    GROUP BY c.condition_text
    ORDER BY total_readings DESC
""")

insight_6.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}insight_conditions/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.insight_conditions
    USING DELTA
    LOCATION '{GOLD_PATH}insight_conditions/'
""")

print("🌍 Insight 6: Weather Condition Distribution:")
display(insight_6.limit(15))

In [0]:
# ── Insight 7: Humidity Levels by Region ──────────────────
insight_7 = spark.sql(f"""
    SELECT
        l.country,
        l.region_category,
        ROUND(AVG(f.humidity), 2)   AS avg_humidity,
        ROUND(MAX(f.humidity), 2)   AS max_humidity,
        ROUND(MIN(f.humidity), 2)   AS min_humidity,
        c.humidity_level,
        COUNT(*)                    AS reading_count
    FROM {CATALOG}.gold.fact_weather f
    JOIN {CATALOG}.gold.dim_location l
        ON f.weather_record_id = l.weather_record_id
    JOIN {CATALOG}.gold.dim_condition c
        ON f.weather_record_id = c.weather_record_id
    WHERE f.is_outlier = false
    GROUP BY l.country, l.region_category, c.humidity_level
    ORDER BY avg_humidity DESC
""")

insight_7.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}insight_humidity/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.insight_humidity
    USING DELTA
    LOCATION '{GOLD_PATH}insight_humidity/'
""")

print("💧 Insight 7: Top 10 Most Humid Regions:")
display(insight_7.limit(10))

In [0]:
# ── Insight 8: Extreme Weather Events by Country ──────────
# for example if UAE has extreme_events = 15, it means 15 weather readings from UAE had at least one sensor value outside these normal ranges.
# Important to note — these aren't natural disasters. They're either:

# Real extreme weather (genuine heatwaves, storms)
# Sensor errors (faulty equipment sending 999°C)
# The fake outliers we injected in Bronze (temp=999, humidity=150, uv=-5)

insight_8 = spark.sql(f"""
    SELECT
        l.country,
        COUNT(*)                                        AS total_readings,
        SUM(CASE WHEN f.is_outlier = true THEN 1 
                 ELSE 0 END)                            AS extreme_events,
        ROUND(SUM(CASE WHEN f.is_outlier = true 
                       THEN 1 ELSE 0 END) * 100.0 
              / COUNT(*), 2)                            AS extreme_pct,
        ROUND(AVG(f.wind_kph), 2)                      AS avg_wind_kph,
        ROUND(MAX(f.wind_kph), 2)                      AS max_wind_kph
    FROM {CATALOG}.gold.fact_weather f
    JOIN {CATALOG}.gold.dim_location l
        ON f.weather_record_id = l.weather_record_id
    GROUP BY l.country
    ORDER BY extreme_events DESC
""")

insight_8.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}insight_extreme_weather/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.insight_extreme_weather
    USING DELTA
    LOCATION '{GOLD_PATH}insight_extreme_weather/'
""")

print("⚡ Insight 8: Top 10 Countries with Most Extreme Weather:")
display(insight_8.limit(10))

In [0]:
# ── Insight 9: Visibility Trends by Weather Condition ─────
insight_9 = spark.sql(f"""
    SELECT
        c.condition_text,
        c.visibility_category,
        ROUND(AVG(f.visibility_km), 2)  AS avg_visibility_km,
        ROUND(MIN(f.visibility_km), 2)  AS min_visibility_km,
        ROUND(MAX(f.visibility_km), 2)  AS max_visibility_km,
        ROUND(AVG(f.cloud), 2)          AS avg_cloud_cover,
        COUNT(*)                        AS reading_count
    FROM {CATALOG}.gold.fact_weather f
    JOIN {CATALOG}.gold.dim_condition c
        ON f.weather_record_id = c.weather_record_id
    WHERE f.is_outlier = false
    GROUP BY c.condition_text, c.visibility_category
    ORDER BY avg_visibility_km ASC
""")

insight_9.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}insight_visibility/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.insight_visibility
    USING DELTA
    LOCATION '{GOLD_PATH}insight_visibility/'
""")

print("👁️ Insight 9: Visibility Trends by Weather Condition:")
display(insight_9.limit(10))

In [0]:
# ── Insight 10: Air Quality vs Cloud Cover ─────────────────
insight_10 = spark.sql(f"""
    SELECT
        CASE
            WHEN f.cloud = 0          THEN 'Clear (0%)'
            WHEN f.cloud <= 25        THEN 'Mostly Clear (1-25%)'
            WHEN f.cloud <= 50        THEN 'Partly Cloudy (26-50%)'
            WHEN f.cloud <= 75        THEN 'Mostly Cloudy (51-75%)'
            ELSE                           'Overcast (76-100%)'
        END                                     AS cloud_category,
        ROUND(AVG(f.air_quality_pm2_5), 2)      AS avg_pm2_5,
        ROUND(AVG(f.air_quality_pm10), 2)       AS avg_pm10,
        ROUND(AVG(f.air_quality_ozone), 2)      AS avg_ozone,
        ROUND(AVG(f.air_quality_us_epa_index), 2) AS avg_epa_index,
        COUNT(*)                                AS reading_count
    FROM {CATALOG}.gold.fact_weather f
    WHERE f.is_outlier = false
    GROUP BY cloud_category
    ORDER BY avg_pm2_5 DESC
""")

insight_10.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}insight_cloud_air_quality/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.insight_cloud_air_quality
    USING DELTA
    LOCATION '{GOLD_PATH}insight_cloud_air_quality/'
""")

print("☁️ Insight 10: Air Quality vs Cloud Cover:")
display(insight_10)

In [0]:
# ── Insight 11: Temperature Variation Day vs Night ─────────
insight_11 = spark.sql(f"""
    SELECT
        l.country,
        d.is_day,
        ROUND(AVG(f.temperature_celsius), 2)    AS avg_temp,
        ROUND(MAX(f.temperature_celsius), 2)    AS max_temp,
        ROUND(MIN(f.temperature_celsius), 2)    AS min_temp,
        ROUND(AVG(f.humidity), 2)               AS avg_humidity,
        COUNT(*)                                AS reading_count
    FROM {CATALOG}.gold.fact_weather f
    JOIN {CATALOG}.gold.dim_location l
        ON f.weather_record_id = l.weather_record_id
    JOIN {CATALOG}.gold.dim_date d
        ON f.weather_record_id = d.weather_record_id
    WHERE f.is_outlier = false
    GROUP BY l.country, d.is_day
    ORDER BY l.country, d.is_day
""")

insight_11.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}insight_day_night_temp/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.insight_day_night_temp
    USING DELTA
    LOCATION '{GOLD_PATH}insight_day_night_temp/'
""")

print("🌙 Insight 11: Temperature Variation Day vs Night (Top 10 Countries):")
display(insight_11.limit(20))

In [0]:
# ── Insight 12: Pressure Analysis by Region ───────────────
insight_12 = spark.sql(f"""
    SELECT
        l.country,
        l.region_category,
        ROUND(AVG(f.pressure_mb), 2)    AS avg_pressure_mb,
        ROUND(MAX(f.pressure_mb), 2)    AS max_pressure_mb,
        ROUND(MIN(f.pressure_mb), 2)    AS min_pressure_mb,
        ROUND(STDDEV(f.pressure_mb), 2) AS pressure_stddev,
        COUNT(*)                        AS reading_count
    FROM {CATALOG}.gold.fact_weather f
    JOIN {CATALOG}.gold.dim_location l
        ON f.weather_record_id = l.weather_record_id
    WHERE f.is_outlier = false
    GROUP BY l.country, l.region_category
    ORDER BY avg_pressure_mb DESC
""")

insight_12.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}insight_pressure/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.insight_pressure
    USING DELTA
    LOCATION '{GOLD_PATH}insight_pressure/'
""")

print("🌡️ Insight 12: Pressure Analysis by Region (Top 10):")
display(insight_12.limit(10))

In [0]:
# ── Insight 13: Best Weather Countries ────────────────────
# Low UV + Good air quality + Mild temp + Low wind = Best weather
# Score is calculated — lower is better

insight_13 = spark.sql(f"""
    SELECT
        l.country,
        ROUND(AVG(f.temperature_celsius), 2)    AS avg_temp,
        ROUND(AVG(f.uv_index), 2)               AS avg_uv,
        ROUND(AVG(f.air_quality_pm2_5), 2)      AS avg_pm2_5,
        ROUND(AVG(f.wind_kph), 2)               AS avg_wind,
        ROUND(AVG(f.humidity), 2)               AS avg_humidity,
        -- Weather score: penalize extreme temps, high UV, bad air, strong wind
        ROUND(
            ABS(AVG(f.temperature_celsius) - 22) +
            AVG(f.uv_index) * 2 +
            AVG(f.air_quality_pm2_5) / 10 +
            AVG(f.wind_kph) / 5,
        2)                                      AS weather_score,
        COUNT(*)                                AS reading_count
    FROM {CATALOG}.gold.fact_weather f
    JOIN {CATALOG}.gold.dim_location l
        ON f.weather_record_id = l.weather_record_id
    WHERE f.is_outlier = false
    GROUP BY l.country
    HAVING COUNT(*) > 100
    ORDER BY weather_score ASC
""")

insight_13.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}insight_best_weather/")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.insight_best_weather
    USING DELTA
    LOCATION '{GOLD_PATH}insight_best_weather/'
""")

print("🏆 Insight 13: Top 10 Countries with BEST Weather:")
display(insight_13.limit(10))

In [0]:
# ── Final Gold Layer Verification ─────────────────────────
all_tables = [
    "dim_location",
    "dim_condition",
    "dim_astronomy",
    "dim_date",
    "fact_weather",
    "insight_top_temperatures",
    "insight_air_quality",
    "insight_rainfall",
    "insight_wind",
    "insight_uv_risk",
    "insight_conditions",
    "insight_humidity",
    "insight_extreme_weather",
    "insight_visibility",
    "insight_cloud_air_quality",
    "insight_day_night_temp",
    "insight_pressure",
    "insight_best_weather"
]

print(f"📊 Gold Layer — Final Verification Report")
print(f"{'='*55}")

total_rows = 0
for table in all_tables:
    count = spark.sql(
        f"SELECT COUNT(*) AS cnt FROM {CATALOG}.gold.{table}"
    ).collect()[0]["cnt"]
    cols = len(spark.table(f"{CATALOG}.gold.{table}").columns)
    print(f"   ✅ {table:<35} → {count:>7} rows, {cols} cols")
    total_rows += count

print(f"\n{'='*55}")
print(f"   Total tables created  : {len(all_tables)}")
print(f"   Dim tables            : 4")
print(f"   Fact tables           : 1")
print(f"   Business insights     : 13")
print(f"   Total rows across all : {total_rows}")
print(f"\n🎉 Gold Layer COMPLETE ✓")

In [0]:
%sql
SHOW TABLES IN global_weather_catalog.gold;

In [0]:
# ── Complete Pipeline Summary ──────────────────────────────
print(f"🎉 GLOBAL WEATHER PIPELINE — COMPLETE!")
print(f"{'='*55}")
print(f"\n📦 Infrastructure:")
print(f"   S3 Bucket         : s3://global-weather-pipeline")
print(f"   Kinesis Stream    : global-weather-data-stream")
print(f"   Firehose          : global-weather-firehose")
print(f"   Glue DB           : global-weather-glue-db")
print(f"   Lambda            : global-weather-lambda")
print(f"   Unity Catalog     : global_weather_catalog")

print(f"\n🥉 Bronze Layer:")
bronze = spark.sql(f"SELECT COUNT(*) AS cnt FROM {CATALOG}.bronze.bronze_weather").collect()[0]["cnt"]
print(f"   Table             : bronze_weather")
print(f"   Rows              : {bronze}")
print(f"   Source            : Raw CSV → Delta")

print(f"\n🥈 Silver Layer:")
silver = spark.sql(f"SELECT COUNT(*) AS cnt FROM {CATALOG}.silver.weather_silver").collect()[0]["cnt"]
print(f"   Table             : weather_silver")
print(f"   Rows              : {silver}")
print(f"   Transformations   : Type casting, null handling, deduplication,")
print(f"                       outlier flagging, derived columns")

print(f"\n🥇 Gold Layer:")
fact = spark.sql(f"SELECT COUNT(*) AS cnt FROM {CATALOG}.gold.fact_weather").collect()[0]["cnt"]
print(f"   Dim tables        : 4 (location, condition, astronomy, date)")
print(f"   Fact table        : fact_weather ({fact} rows)")
print(f"   Business insights : 13 insight tables")

print(f"\n✅ Pipeline Status   : SUCCESSFUL ✓")
print(f"{'='*55}")